<a href="https://colab.research.google.com/github/DominicVerschoor/TVA_Strategic-Voting/blob/Happiness/Happiness_metrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# For one winner

In [ ]:
import numpy as np

def rank_based_happiness(preferences, winner):
    # happiness to be between 0 (least happy, when the winner is the worst choice) and 1 (happiest, when the top preference wins).
    m = len(preferences[0])  # Number of alternatives
    happiness = [(m - voter_prefs.index(winner)) / (m) for voter_prefs in preferences]
    return happiness

In [ ]:
def utility_based_happiness(preferences, winner, utility='linear'):
    # Linear behaves like the rank based happiness but exponential punishes more far away rankings
    m = len(preferences[0])
    happiness = []

    for voter_prefs in preferences:
        rank = voter_prefs.index(winner)
        if utility == 'linear':
            happiness.append((m - rank) / m)
        elif utility == 'exponential':
            happiness.append((2 ** (m - rank) - 1) / (2 ** m - 1))

    return happiness

In [ ]:
def borda_score_happiness(preferences, winner):
    # Like the Borda score method
    m = len(preferences[0])
    happiness = []

    for voter_prefs in preferences:
        borda_scores = {alt: m - voter_prefs.index(alt) - 1 for alt in voter_prefs}
        happiness.append(borda_scores[winner] / max(borda_scores.values()))

    return happiness

In [ ]:
preferences = [['A', 'B', 'C'], ['B', 'C', 'A'], ['C', 'A', 'B']]
winner = 'B'

print("Rank-Based Happiness:", rank_based_happiness(preferences, winner))
print("Utility-Based Happiness (Linear):", utility_based_happiness(preferences, winner, utility='linear'))
print("Utility-Based Happiness (Exponential):", utility_based_happiness(preferences, winner, utility='exponential'))
print("Borda Score Happiness:", borda_score_happiness(preferences, winner))

Rank-Based Happiness: [0.6666666666666666, 1.0, 0.3333333333333333]
Utility-Based Happiness (Linear): [0.6666666666666666, 1.0, 0.3333333333333333]
Utility-Based Happiness (Exponential): [0.42857142857142855, 1.0, 0.14285714285714285]
Borda Score Happiness: [0.5, 1.0, 0.0]


# For position of each candidate

In [1]:
def create_symmetric_array(n):
    if n < 1:
        return []
    first_half = list(range(n, 0, -2))
    second_half = first_half[::-1]
    if len(first_half) + len(second_half) > n:
        second_half = second_half[1:]
    return first_half + second_half

In [2]:
from scipy.stats import kendalltau

def compute_happiness(preferences, final_ranking): #https://link.springer.com/chapter/10.1007/978-3-322-80613-0_7
    n = len(final_ranking)  # Number of candidates
    happiness_scores = []
    array = create_symmetric_array(n) # get the weight array
    max_score = sum(x * n for x in array) # Max score

    for voter in preferences:
        if(voter[0]==final_ranking[0]):
            pos_score = max_score
        else:
          # Compute Positional Satisfaction Score
          pos_score = sum(array[i]*(n - abs(voter.index(c) - final_ranking.index(c))) for i, c in enumerate(voter))

        happiness = pos_score/max_score # Normalization step
        happiness_scores.append(happiness)

    return happiness_scores

# Example input
preferences = [['A', 'B', 'C','D'], ['B', 'C', 'A','D'], ['D','C', 'A', 'B']] #Put a dataframe later on (for analyzing)
winner = ['B', 'C', 'A','D'] # Final ranking

# Compute happiness
happiness_scores = compute_happiness(preferences, winner)
print(happiness_scores)

[0.75, 1.0, 0.5]


Voting for 2

In [6]:
from scipy.stats import kendalltau

def compute_happiness(preferences, final_ranking): # Vote for 2
    n = len(final_ranking)  # Number of candidates
    happiness_scores = []
    array = create_symmetric_array(n) # get the weight array
    array[1] = array[0]
    max_score = sum(x * n for x in array) # Max score

    for voter in preferences:
        if(voter[0]==final_ranking[0] or voter[1]==final_ranking[0]):
            pos_score = max_score
        else:
          # Compute Positional Satisfaction Score
          pos_score = sum(array[i]*(n - abs(voter.index(c) - final_ranking.index(c))) for i, c in enumerate(voter))

        happiness = pos_score/max_score # Normalization step
        happiness_scores.append(happiness)

    return happiness_scores

# Example input
preferences = ['A', 'B'], ['B', 'C'], ['C','B'], ['D','C'], ['A', 'B'] #Put a dataframe later on (for analyzing)
winner = ['B', 'C', 'A', 'D'] # Final ranking

# Compute happiness
happiness_scores = compute_happiness(preferences, winner)
print(happiness_scores)

[1.0, 1.0, 1.0, 0.35714285714285715, 1.0]
